In [1]:

import pandas as pd

matches = pd.read_csv('/workspaces/curling_data/output/matches.csv')
ends = pd.read_csv('/workspaces/curling_data/output/ends.csv')
events = pd.read_csv('/workspaces/curling_data/output/events.csv')

# Find still-null CWC matches after the previous fix
still_null = matches[
    (matches['team1_final_score'].isna() | matches['team2_final_score'].isna()) &
    matches['event_id'].isin([147, 160, 165, 174])
].copy()
print(f"Still-NULL CWC matches: {len(still_null)}")

# Do any still-null matches have ANY ends?
cwc_ends = ends[ends['event_id'].isin([147, 160, 165, 174])].copy()
cwc_ends['pair'] = list(zip(cwc_ends['event_id'], cwc_ends['match_id']))
null_pairs = set(zip(still_null['event_id'], still_null['match_id']))
still_null_with_ends = cwc_ends[cwc_ends['pair'].isin(null_pairs)]
print(f"Still-null matches that DO have ends: {still_null_with_ends[['event_id','match_id']].drop_duplicates().shape[0]}")

# Round breakdown comparison
fixed_cwc = matches[matches['event_id'].isin([147, 160, 165, 174]) & matches['team1_final_score'].notna()]
print("\nRound types in CWC matches WITH scores (top 10):")
print(fixed_cwc['round'].value_counts().head(10))
print("\nRound types in still-NULL CWC matches (top 10):")
print(still_null['round'].value_counts().head(10))


Still-NULL CWC matches: 78
Still-null matches that DO have ends: 0

Round types in CWC matches WITH scores (top 10):
round
Round Robin Session 12    17
Round Robin Session 5     17
Round Robin Session 15    15
Round Robin Session 7     15
Round Robin Session 10    14
Round Robin Session 3     14
Round Robin Session 13    11
Round Robin Session 9     11
Round Robin Session 14    11
Round Robin Session 8      9
Name: count, dtype: int64

Round types in still-NULL CWC matches (top 10):
round
Round Robin Session 4     13
Round Robin Session 11    11
Round Robin Session 6     10
Round Robin Session 2      9
Round Robin Session 8      7
Round Robin Session 13     6
Round Robin Session 14     5
Round Robin Session 9      5
Round Robin Session 1      4
Round Robin Session 10     2
Name: count, dtype: int64


In [2]:

# Look at event 174 (CWC Leg 1) - compare with-ends and without-ends matches
e174_matches = matches[matches['event_id'] == 174].copy()
e174_ends = ends[ends['event_id'] == 174].copy()

# Tag which have ends
matches_with_ends = set(e174_ends['match_id'].unique())
e174_matches['has_ends'] = e174_matches['match_id'].isin(matches_with_ends)
e174_matches['has_score'] = e174_matches['team1_final_score'].notna()

print("Event 174 - match breakdown:")
print(e174_matches.groupby(['has_score', 'has_ends']).size().reset_index(name='count'))

print("\nSample of matches WITH ends (first 5):")
print(e174_matches[e174_matches['has_ends']].head(5)[
    ['match_id','round','team1_code','team2_code','team1_final_score','team2_final_score','has_ends']
].to_string(index=False))

print("\nSample of matches WITHOUT ends (first 5):")
print(e174_matches[~e174_matches['has_ends']].head(5)[
    ['match_id','round','team1_code','team2_code','team1_final_score','team2_final_score','has_ends']
].to_string(index=False))

# How many ends per match for those WITH ends?
end_counts = e174_ends.groupby('match_id').size().reset_index(name='n_ends')
print(f"\nEnd counts for CWC Leg1 matches: min={end_counts['n_ends'].min()}, max={end_counts['n_ends'].max()}, median={end_counts['n_ends'].median()}")
print(end_counts['n_ends'].value_counts().sort_index())


Event 174 - match breakdown:
   has_score  has_ends  count
0      False     False     25
1       True      True     50

Sample of matches WITH ends (first 5):
 match_id                  round team1_code team2_code  team1_final_score  team2_final_score  has_ends
        1                  Final        CAN        SWE                7.0                3.0      True
        2 Round Robin Session 15        CAN        USA                8.0                2.0      True
        3 Round Robin Session 15        SWE        SCO                8.0                1.0      True
        4 Round Robin Session 15        CHN        JPN                5.0                4.0      True
        5 Round Robin Session 15        RUS        KOR               10.0                1.0      True

Sample of matches WITHOUT ends (first 5):
 match_id                  round team1_code team2_code  team1_final_score  team2_final_score  has_ends
       51                  Final        USA        CAN                NaN    

In [4]:

# Check team overlap between with-ends and without-ends in event 174
e174 = matches[matches['event_id'] == 174].copy()
e174_ends_matchids = set(ends[ends['event_id'] == 174]['match_id'])
e174['has_ends'] = e174['match_id'].isin(e174_ends_matchids)

with_ends_teams = set(e174[e174['has_ends']]['team1_code'].tolist() + e174[e174['has_ends']]['team2_code'].tolist())
without_ends_teams = set(e174[~e174['has_ends']]['team1_code'].tolist() + e174[~e174['has_ends']]['team2_code'].tolist())
print("Teams ONLY in without-ends:", sorted(without_ends_teams - with_ends_teams))
print("Teams in BOTH groups:", sorted(with_ends_teams & without_ends_teams))

# Are the same team pairs appearing in both groups?
e174_with = e174[e174['has_ends']].copy()
e174_without = e174[~e174['has_ends']].copy()
e174_with['pair'] = e174_with.apply(lambda r: frozenset([r['team1_code'],r['team2_code']]), axis=1)
e174_without['pair'] = e174_without.apply(lambda r: frozenset([r['team1_code'],r['team2_code']]), axis=1)
shared_pairs = set(e174_with['pair']) & set(e174_without['pair'])
print(f"\nSame matchups in both groups: {len(shared_pairs)}")
print("First few shared:", list(shared_pairs)[:5])

# Is there a pattern in match_id ranges?
print(f"\nWith-ends match_id range: {e174_with['match_id'].min()} - {e174_with['match_id'].max()}")
print(f"Without-ends match_id range: {e174_without['match_id'].min()} - {e174_without['match_id'].max()}")


Teams ONLY in without-ends: []
Teams in BOTH groups: ['CAN', 'CHN', 'KOR', 'NOR', 'RUS', 'SUI', 'SWE', 'USA']

Same matchups in both groups: 7
First few shared: [frozenset({'RUS', 'CAN'}), frozenset({'NOR', 'SWE'}), frozenset({'USA', 'KOR'}), frozenset({'CAN', 'NOR'}), frozenset({'USA', 'CHN'})]

With-ends match_id range: 1 - 50
Without-ends match_id range: 51 - 75


In [9]:

# Download a CWC PDF and inspect its text structure
import urllib.request, pdfplumber, os

url = "https://curlit.com/PDF/CWC2018-19_Leg1_ResultsBook.pdf"
local_path = "/tmp/cwc_leg1.pdf"

if not os.path.exists(local_path):
    print(f"Downloading {url} ...")
    urllib.request.urlretrieve(url, local_path)
    print("Done.")
else:
    print("Already downloaded.")

with pdfplumber.open(local_path) as pdf:
    print(f"Total pages: {len(pdf.pages)}")
    # Show text of first 3 pages to understand the format
    for i in range(min(3, len(pdf.pages))):
        print(f"\n=== PAGE {i} ===")
        print(repr(pdf.pages[i].extract_text()[:2000]))


Matches where final_score == last ends score_after: 3072 / 3325
Mismatches: 253
First 10 mismatches:
 event_id  match_id  team1_final_score  team1_score_after  team2_final_score  team2_score_after
        1         2                4.0                  4                7.0                  5
        1         4                7.0                  6                6.0                  6
        1         7                6.0                  5                5.0                  5
        1        11                7.0                  4                4.0                  4
        1        16                5.0                  5                6.0                  4
        1        28                7.0                  7                5.0                  3
        1        41                5.0                  2                4.0                  4
        1        43                5.0                  5                6.0                  5
        4         2               1

In [14]:

# Is the scraper invoked with BOTH result_book_url AND result_summary_url?
# Check how load_result_urls produces PDF paths
result_urls = pd.read_csv('/workspaces/curling_data/output/result_urls.csv')
cwc_rows = result_urls[result_urls['tournament_name'].str.contains('Curling World Cup', na=False)]
print("CWC rows in result_urls.csv:")
print(cwc_rows[['tournament_name','year','result_book_url','result_summary_url']].to_string(index=False))


CWC rows in result_urls.csv:
                      tournament_name   year                                         result_book_url                                         result_summary_url
Curling World Cup 2018/19 Grand Final 2018.0 https://curlit.com/PDF/CWC2018-19_Final_ResultsBook.pdf https://curlit.com/PDF/CWC2018-19_Final_ResultsSummary.pdf
      Curling World Cup 2018/19 Leg 3 2018.0  https://curlit.com/PDF/CWC2018-19_Leg3_ResultsBook.pdf  https://curlit.com/PDF/CWC2018-19_Leg3_ResultsSummary.pdf
      Curling World Cup 2018/19 Leg 2 2018.0  https://curlit.com/PDF/CWC2018-19_Leg2_ResultsBook.pdf  https://curlit.com/PDF/CWC2018-19_Leg2_ResultsSummary.pdf
      Curling World Cup 2018/19 Leg 1 2018.0  https://curlit.com/PDF/CWC2018-19_Leg1_ResultsBook.pdf  https://curlit.com/PDF/CWC2018-19_Leg1_ResultsSummary.pdf


In [10]:

# Reload matches (saved by previous fix cell) and investigate still-null matches
matches2 = pd.read_csv('/workspaces/curling_data/output/matches.csv')
ends2 = pd.read_csv('/workspaces/curling_data/output/ends.csv')

# Find still-null CWC matches
still_null = matches2[
    (matches2['team1_final_score'].isna() | matches2['team2_final_score'].isna()) &
    matches2['event_id'].isin([147, 160, 165, 174])
].copy()

print(f"Still-NULL CWC matches: {len(still_null)}")

# Check what ends these matches have
cwc_ends = ends2[ends2['event_id'].isin([147, 160, 165, 174])]
null_match_ids_with_event = set(zip(still_null['event_id'], still_null['match_id']))

# Do any still-null matches have ANY ends?
cwc_ends['pair'] = list(zip(cwc_ends['event_id'], cwc_ends['match_id']))
still_null_with_ends = cwc_ends[cwc_ends['pair'].isin(null_match_ids_with_event)]
print(f"Still-null matches that DO have ends: {still_null_with_ends[['event_id','match_id']].drop_duplicates().shape[0]}")

# Show a few still-null rows
print("\nSample of still-null CWC matches:")
print(still_null.head(10)[['event_id','match_id','round','team1_code','team2_code','team1_final_score','team2_final_score']].to_string(index=False))

# And look at the matches that WERE fixed for comparison — what rounds do they have?
fixed_cwc = matches2[
    matches2['event_id'].isin([147, 160, 165, 174]) &
    matches2['team1_final_score'].notna()
]
print("\nRound types in CWC matches WITH scores:")
print(fixed_cwc['round'].value_counts().head(10))
print("\nRound types in still-NULL CWC matches:")
print(still_null['round'].value_counts().head(10))


Still-NULL CWC matches: 78
Still-null matches that DO have ends: 0

Sample of still-null CWC matches:
 event_id  match_id                  round team1_code team2_code  team1_final_score  team2_final_score
      147        51        Final - Sheet C       CAN1        NOR                NaN                NaN
      147        52 Round Robin Session 15        NOR       SUI2                NaN                NaN
      147        53 Round Robin Session 14       CAN2        RUS                NaN                NaN
      147        54 Round Robin Session 13       CAN1       SUI1                NaN                NaN
      147        55 Round Robin Session 13        CHN        USA                NaN                NaN
      147        56 Round Robin Session 11        RUS        NOR                NaN                NaN
      147        57 Round Robin Session 11       SUI2       CAN2                NaN                NaN
      147        58 Round Robin Session 11       SUI1        CHN          